# Kiểm tra chất lượng dữ liệu collector

Đợt kỹ thuật 12/11/2026: PEPE, BRETT, HEGE. Chỉ kiểm tra độ phủ và truy nguyên; các pool này chưa phải mẫu nghiên cứu, không đo target hoặc chỉnh ngưỡng từ tập test.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import duckdb

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
FOLDER = ROOT / "data/interim/collector_check/ohlcv_online"
data = pd.read_csv(FOLDER / "ohlcv.csv", parse_dates=["ts"])
report = json.loads((FOLDER / "report.json").read_text())
print(f"{len(data):,} dòng OHLCV")


## Độ phủ theo chuỗi và loại nến

Không có nến không có nghĩa là volume bằng 0. Báo cáo giữ số khoảng thiếu để nhóm kiểm tra thêm.


In [ ]:
coverage = pd.DataFrame(report["coverage"])
coverage[["chain", "timeframe", "candles", "expected_intervals", "missing_intervals", "coverage_complete", "first_ts", "last_ts"]]


## Trùng khóa và trường thiếu


In [ ]:
keys = ["chain", "pool_address", "timeframe", "ts"]
assert data.duplicated(keys).sum() == 0
assert data["source"].notna().all() and data["raw_ref"].notna().all()
data.isna().sum().to_frame("số dòng thiếu")


## Đối chiếu CSV, DuckDB và Parquet


In [ ]:
with duckdb.connect(str(FOLDER / "collection.duckdb"), read_only=True) as db:
    n_db = db.execute("SELECT count(*) FROM ohlcv").fetchone()[0]
    n_parquet = db.execute("SELECT count(*) FROM read_parquet(?)", [str(FOLDER / "ohlcv.parquet")]).fetchone()[0]
    assert n_db == n_parquet == len(data)
print("Ba định dạng cùng số dòng:", n_db)


## Theo một dòng về phản hồi gốc

Metadata có SHA-256 và request đã che key. Không sửa file raw.


In [ ]:
example = data.iloc[0]
raw_path = ROOT / example["raw_ref"]
metadata = json.loads(raw_path.with_suffix(".meta.json").read_text())
import hashlib
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == metadata["sha256"]
metadata


Chạy thêm `.venv/bin/python scripts/check_collection.py` ở Terminal để đối chiếu byte của CSV replay và 8 signature Solana. Manual validation trên explorer vẫn là bước riêng.
